In [1]:
import chromadb
import ollama
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.PersistentClient(path="../chroma_db")
collection = client.get_collection(name="knee_passages")

print("Veritabanindaki pasaj sayisi:", collection.count())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Veritabanindaki pasaj sayisi: 6


In [2]:
def ask(question, k=3):
    # 1) RETRIEVAL: Soruya en yakin pasajlari bul
    question_vector = model.encode(question).tolist()
    results = collection.query(query_embeddings=[question_vector], n_results=k)

    # 2) AUGMENTATION: Pasajlari numaralandirip tek bir metinde birlestir
    context = ""
    for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0]), start=1):
        context += f"[{i}] ({meta['source']}) {doc}\n"

    prompt = f"""Answer the question using ONLY the passages below.
After your answer, write the number of the passage you used, like [1].
If the passages do not contain the answer, reply exactly:
"I don't know, please consult your healthcare team."

Passages:
{context}
Question: {question}
Answer:"""

    # 3) GENERATION: Modele sor
    response = ollama.chat(model="llama3.2", messages=[{"role": "user", "content": prompt}])

    print("BULUNAN PASAJLAR:")
    print(context)
    print("MODELIN CEVABI:")
    print(response["message"]["content"])

In [3]:
ask("When can I walk after knee surgery?")


BULUNAN PASAJLAR:
[1] (NHS (ornek)) Most people stay in hospital for 1 to 3 days after knee replacement surgery.
[2] (NHS (ornek)) Most patients begin moving around with a walker the day after the operation.
[3] (MedlinePlus (ornek)) Physical therapy helps strengthen the muscles around your new knee.

MODELIN CEVABI:
The day after the operation.

[2]


In [4]:
ask("What should I eat before knee surgery?")

BULUNAN PASAJLAR:
[1] (NIH NIAMS (ornek)) Knee replacement is a common procedure for severe arthritis.
[2] (MedlinePlus (ornek)) Physical therapy helps strengthen the muscles around your new knee.
[3] (NHS (ornek)) Most people stay in hospital for 1 to 3 days after knee replacement surgery.

MODELIN CEVABI:
I don't know, please consult your healthcare team.

[1]


In [5]:
ask("How much ibuprofen should I take after knee surgery?")

BULUNAN PASAJLAR:
[1] (NHS (ornek)) Most people stay in hospital for 1 to 3 days after knee replacement surgery.
[2] (NIH NIAMS (ornek)) Knee replacement is a common procedure for severe arthritis.
[3] (MedlinePlus (ornek)) Blood clots are a possible risk after knee replacement surgery.

MODELIN CEVABI:
I don't know, please consult your healthcare team.

[3]


In [6]:
new_passage = "After surgery, you may be given painkillers such as paracetamol or ibuprofen to help manage your pain."

collection.upsert(
    ids=["p7"],
    documents=[new_passage],
    embeddings=[model.encode(new_passage).tolist()],
    metadatas=[{"source": "NHS (ornek)", "topic": "medication"}],
)

print("Pasaj sayisi:", collection.count())

ask("How much ibuprofen should I take after knee surgery?")

Pasaj sayisi: 7
BULUNAN PASAJLAR:
[1] (NHS (ornek)) After surgery, you may be given painkillers such as paracetamol or ibuprofen to help manage your pain.
[2] (NHS (ornek)) Most people stay in hospital for 1 to 3 days after knee replacement surgery.
[3] (NIH NIAMS (ornek)) Knee replacement is a common procedure for severe arthritis.

MODELIN CEVABI:
I don't know, please consult your healthcare team.

[1]


In [7]:
ask("Can I take ibuprofen after knee surgery?")

BULUNAN PASAJLAR:
[1] (NHS (ornek)) After surgery, you may be given painkillers such as paracetamol or ibuprofen to help manage your pain.
[2] (NHS (ornek)) Most people stay in hospital for 1 to 3 days after knee replacement surgery.
[3] (NIH NIAMS (ornek)) Knee replacement is a common procedure for severe arthritis.

MODELIN CEVABI:
[1]

After your surgery, you may be given painkillers such as paracetamol or ibuprofen to help manage your pain.


In [8]:
import re

MEDICATION_KEYWORDS = [
    "ibuprofen", "paracetamol", "acetaminophen", "aspirin", "painkiller", "painkillers",
    "medication", "medications", "medicine", "drug", "drugs", "dose", "dosage",
    "pill", "pills", "tablet", "tablets", "mg", "antibiotic", "antibiotics", "blood thinner",
]

EMERGENCY_KEYWORDS = [
    "emergency", "chest pain", "can't breathe", "cannot breathe",
    "heavy bleeding", "unconscious", "fainted",
]

SAFETY_MESSAGE = "I can't answer questions about medications or emergencies. Please consult your healthcare team. In an emergency, call emergency services immediately."


def is_unsafe(question):
    q = question.lower()
    for keyword in MEDICATION_KEYWORDS + EMERGENCY_KEYWORDS:
        if re.search(r"\b" + re.escape(keyword) + r"\b", q):
            return True
    return False


def ask_safe(question, k=3):
    # 0) GUVENLIK KONTROLU: Arama ve LLM'den ONCE
    if is_unsafe(question):
        print("GUVENLIK KURALI DEVREDE: Soru arama yapilmadan reddedildi.")
        print("CEVAP:")
        print(SAFETY_MESSAGE)
        return
    # Guvenliyse normal RAG akisi
    ask(question, k)

In [9]:
ask_safe("Can I take ibuprofen after knee surgery?")
print("\n" + "=" * 60 + "\n")
ask_safe("I have chest pain after my surgery, what should I do?")
print("\n" + "=" * 60 + "\n")
ask_safe("When can I walk after knee surgery?")

GUVENLIK KURALI DEVREDE: Soru arama yapilmadan reddedildi.
CEVAP:
I can't answer questions about medications or emergencies. Please consult your healthcare team. In an emergency, call emergency services immediately.


GUVENLIK KURALI DEVREDE: Soru arama yapilmadan reddedildi.
CEVAP:
I can't answer questions about medications or emergencies. Please consult your healthcare team. In an emergency, call emergency services immediately.


BULUNAN PASAJLAR:
[1] (NHS (ornek)) Most people stay in hospital for 1 to 3 days after knee replacement surgery.
[2] (NHS (ornek)) Most patients begin moving around with a walker the day after the operation.
[3] (MedlinePlus (ornek)) Physical therapy helps strengthen the muscles around your new knee.

MODELIN CEVABI:
The day after the operation.

[2]


ask_safe("Can I take Advil after knee surgery?")

# Deneyden Çıkan Sonuçlar

## 1. İlk RAG prototipi çalıştı
Embedding, ChromaDB ve LLM (llama3.2) ilk kez birleştirildi. Soru vektöre çevrildi, ChromaDB'den en yakın 3 pasaj bulundu ve pasajlar bir prompt içinde modele verildi. Model sadece bu pasajlara dayanarak cevap verdi ve kaynağını gösterdi.

## 2. Veritabanı kalıcı
Bir önceki deneyde eklenen 6 pasaj, Jupyter kapatılıp açıldıktan sonra bile veritabanında duruyordu.

## 3. LLM doğru pasajı seçebiliyor
"Ne zaman yürüyebilirim?" sorusunda arama, yanlış pasajı birinci sıraya koymuştu. LLM üç pasajı da okudu, birinci sıradakine aldanmadı ve asıl cevabı içeren **[2]** numaralı pasajı seçti. Bu, Top-K değerinin 1'den büyük olması gerektiğini de doğruluyor.

## 4. LLM, cevap pasajlarda yoksa çekilebiliyor
"Ameliyattan önce ne yemeliyim?" sorusunda pasajların hiçbiri beslenmeden bahsetmiyordu. Model kendi hafızasından uydurmak yerine **"I don't know, please consult your healthcare team."** dedi.

## 5. Sızdırmazlık riski gerçek
Veritabanına ağrı kesicilerden bahseden bir pasaj eklendiğinde, "İbuprofen alabilir miyim?" sorusuna model pasajı kullanarak cevap verdi. Model kurallara uydu, ama ilaç sorusunu cevaplamış oldu. Bu yüzden ilaç ve acil durum soruları **aramadan ve LLM'den önce** ayrı bir güvenlik kuralıyla reddedilmeli.

## 6. Güvenlik katmanı eklendi
Anahtar kelime listesine dayalı basit bir güvenlik kuralı yazıldı. İlaç ve acil durum soruları arama yapılmadan sabit bir mesajla reddedildi. Normal sorular ise eskisi gibi cevaplandı.

## 7. Anahtar kelime listesi yeterli değil
"Advil alabilir miyim?" sorusu, Advil listede olmadığı için güvenlik kuralından geçti. Ayrıca model kendi hafızasını kullanarak "Advil bir ibuprofen markasıdır" bilgisini ekledi ve pasajdaki "verilebilir" ifadesini "alabilirsiniz" şeklinde abarttı. Bundan çıkan dersler:
- Güvenlik kontrolü LLM'e bırakılmamalı.
- Anahtar kelime listesi; marka adları, yazım hataları ve dolaylı ifadelerle delinebilir. 2. haftada embedding tabanlı ya da LLM tabanlı bir kontrol eklenmeli.
- Değerlendirmede sadece kaynak numarası değil, cevabın pasajla gerçekten örtüşüp örtüşmediği de kontrol edilmeli.
- Reddedilmesi gereken 50 test sorusunun içine zor örnekler (marka adları, yazım hataları, dolaylı ifadeler) eklenmeli.

                     ## hafta da tartışmamız gerekenler;##

güvenlik kuralı nasıl geliştirilebilir?
Ekipçe değerlendirebileceğiniz birkaç seçenek var:

Listeyi genişletmek: Marka adları ve genel ifadeler ("something for pain") eklemek. Kolay ama yine de delinebilir.
Embedding ile kontrol: Bilinen tehlikeli soru örneklerinden oluşan bir liste hazırlayıp gelen soruyu bunlarla karşılaştırmak. Benzerlik yüksekse reddetmek. Bugün öğrendiğin embedding mantığının aynısı. Farklı kelimelerle sorulmuş aynı anlamdaki soruları yakalar.
LLM ile sınıflandırma: Asıl cevaptan önce modele sadece "Bu soru ilaç veya acil durumla mı ilgili? Evet/Hayır" diye sormak.
## Birden fazlasını birleştirmek: En güvenli yaklaşım. !!
  